# Performance- und Machbarkeitstest: A*-Router auf großem Netzwerk

Dieses Notebook evaluiert die Leistung und Lösbarkeit des `AStarRouter` auf dem großen Netzwerk (`large_network.json`) mit einem Planungshorizont von 30 Tagen und 50.000 zufällig generierten Sendungen.

Jede Sendung besitzt hierbei ihre eigene, zufällig generierte Gewichtung der Ziele (Kosten, Zeit und Emissionen).

## Zielsetzung
- Messung der Berechnungsdauer (Gesamtzeit und Zeit pro Sendung).
- Ermittlung der Anzahl und des Prozentanteils unlösbarer Sendungen.
- Auswertung der Bündelung/Konsolidierung unter Berücksichtigung individueller Sendungsgewichte.
- Einbindung des Routers direkt aus `heuristics/dijkstra_router.py`.

## 1. Setup und Imports

In [ ]:
import sys
import time
import random
from pathlib import Path
from collections import defaultdict
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from freight_routing.data_loader import NetworkDataLoader
from freight_routing.data_models import Shipment, ObjectiveWeights, ArcType
from freight_routing.model import TimeExpandedNetwork
from heuristics.dijkstra_router import AStarRouter

## 2. Laden des Netzwerks

In [ ]:
network_path = PROJECT_ROOT / "dataset/large_network.json"
print(f"Lade Netzwerk aus {network_path}...")
t0 = time.time()
network_data = NetworkDataLoader.from_json(network_path)
print(f"Netzwerk in {time.time() - t0:.2f} s geladen.")
network_data.summary()

## 3. Generierung von 50.000 Sendungen mit individuellen Gewichtungen

In [ ]:
# Seed für Reproduzierbarkeit
random.seed(42)
N_SHIPMENTS = 50000
PLANNING_DAYS = 30
DEADLINE = PLANNING_DAYS * 24 * 60  # 30 Tage in Minuten (43.200 min)

hubs_list = list(network_data.hubs.keys())
shipments = []
for i in range(N_SHIPMENTS):
    start = random.choice(hubs_list)
    dest = random.choice(hubs_list)
    while dest == start:
        dest = random.choice(hubs_list)

    # Zufällige Gewichtungen generieren, die sich zu 1.0 aufsummieren
    w_cost = random.uniform(0.01, 1.0)
    w_time = random.uniform(0.01, 1.0)
    w_emissions = random.uniform(0.01, 1.0)
    total_w = w_cost + w_time + w_emissions
    w_cost /= total_w
    w_time /= total_w
    w_emissions /= total_w

    shipments.append(
        Shipment(
            id=f"shipment_{i}",
            start_hub=start,
            end_hub=dest,
            start_time=0,
            deadline=DEADLINE,
            max_price=1000000.0,
            max_emissions=None,
            weight=float(random.randint(1, 10)),
            objective_weights=ObjectiveWeights(
                cost=w_cost,
                time=w_time,
                emissions=w_emissions
            )
        )
    )
print(f"{len(shipments)} Sendungen mit zufälligen Gewichtungen generiert.")

## 4. Durchführung des Performance-Tests

Wir initialisieren den `AStarRouter` und führen die sequentielle Routenplanung mittels `solve_multiple` aus. Jede Sendung wird nach ihren individuellen Gewichtungen geroutet.

**WICHTIG ZUR RUNTIME:** Der A*-Router benötigt dank der statischen Heuristik nur ca. 0.15 Sekunden pro Sendung. Das vollständige Routing aller 50.000 Sendungen dauert daher ca. 2 Stunden.

Für einen schnellen Vorabtest ist `LIMIT_RUN = 1000` voreingestellt (Rechenzeit: ca. 2.5 Minuten). Sie können diesen Parameter auf `None` oder `50000` setzen, um größere Mengen zu testen.

In [ ]:
# Ändern Sie dies auf None oder 50000 für den vollständigen Test.
LIMIT_RUN = 1000

if LIMIT_RUN is not None:
    test_shipments = shipments[:LIMIT_RUN]
    print(f"Führe Test für {LIMIT_RUN} Sendungen aus (geschätzte Zeit: ~2.5 Min)... ")
else:
    test_shipments = shipments
    print(f"Führe VOLLSTÄNDIGEN Test für alle {N_SHIPMENTS} Sendungen aus (geschätzte Zeit: ~2 Stunden)... ")

weights_default = ObjectiveWeights(cost=0.4, emissions=0.4, time=0.2)

# Netzwerk instanziieren und bauen
network = TimeExpandedNetwork(network_data, objective_weights=weights_default)
network.build(planning_days=PLANNING_DAYS, shipments=test_shipments)

router = AStarRouter(objective_weights=weights_default)

t_routing_start = time.time()
routing_result = router.solve_multiple(network)
t_routing_duration = time.time() - t_routing_start

print(f"\nTest abgeschlossen!")
print(f"Gesamtdauer für das Routing: {t_routing_duration:.2f} s")

## 5. Auswertung und Analyse

In [ ]:
total_tested = len(test_shipments)
solved_count = len(routing_result.shipment_routes)
unsolvable_count = total_tested - solved_count
pct_unsolvable = (unsolvable_count / total_tested) * 100
avg_time_per_shipment = t_routing_duration / total_tested

# Analyse der Bündelung/Konsolidierung
transport_arc_shipments = defaultdict(list)
for s_id, route in routing_result.shipment_routes.items():
    for arc in route:
        if arc.arc_type == ArcType.TRANSPORT:
            transport_arc_shipments[arc].append(s_id)

consolidated_shipments = set()
for arc, s_ids in transport_arc_shipments.items():
    if len(s_ids) >= 2:
        for s_id in s_ids:
            consolidated_shipments.add(s_id)

num_consolidated = len(consolidated_shipments)
pct_consolidated = (num_consolidated / solved_count) * 100 if solved_count > 0 else 0.0

print("============================================================")
print(f"                       Testergebnisse                       ")
print("============================================================")
print(f"Gesamtzahl getesteter Sendungen : {total_tested:,}")
print(f"Berechnungsdauer insgesamt      : {t_routing_duration:.2f} s ({t_routing_duration/60:.2f} min)")
print(f"Durchschnittliche Zeit / Sendung : {avg_time_per_shipment:.6f} s")
print(f"Erfolgreich gelöste Sendungen   : {solved_count:,}")
print(f"Anzahl unlösbarer Sendungen     : {unsolvable_count:,}")
print(f"Prozent unlösbarer Sendungen    : {pct_unsolvable:.2f} %")
print(f"Konsolidierte Sendungen         : {num_consolidated:,} von {solved_count:,}")
print(f"Konsolidierungsrate (gelöst)    : {pct_consolidated:.2f} %")
print(f"Konsolidiert (is_consolidated)  : {routing_result.is_consolidated}")
print("============================================================")

if unsolvable_count > 0:
    print("\nBeispiele für unlösbare Sendungen (Diagnostics):")
    for diag in list(routing_result.diagnostics)[:10]:
        print(f" - {diag}")